## Prompt Engineering

Design effective prompts for legal RAG systems using Groq LLM.

> **About Prompt Engineering:** The art of crafting inputs to LLMs to get high-quality, reliable outputs. Good prompts include clear role definitions, context, constraints, and output format instructions. Critical for legal applications where accuracy and citations matter.

### Import Libraries

In [1]:
from llama_index.llms.groq import Groq
from llama_index.core import Settings
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer
import os
import time

### Define Global Model Constant

In [2]:
# Global model - load once, reuse everywhere
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

### Configure API Key

> **Note:** Never hardcode API keys in notebooks! Use environment variables:
> 1. Copy `.env.example` to `.env` in the project root
> 2. Add your Groq API key to `.env`
> 3. Load it with `load_dotenv()`
>
> Get your free API key from: https://console.groq.com/keys

In [3]:
# Load environment variables from .env file
load_dotenv()

# Get API key from environment
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    print("⚠ Warning: GROQ_API_KEY not set!")
    print("Please create a .env file with your API key.")
    print("See .env.example for template.")
else:
    print("✓ GROQ_API_KEY loaded successfully")
    print(f"  Key starts with: {GROQ_API_KEY[:8]}...")

✓ GROQ_API_KEY loaded successfully
  Key starts with: gsk_Q5Ti...


### Initialize Groq LLM

In [4]:
# Define model configuration
# Recommended for free tier (better rate limits):
# - llama-3.1-8b-instant: Fast, efficient, 128K context (BEST for free tier)
# - llama-3.2-11b-vision-preview: Good balance, 128K context
MODEL_NAME = "llama-3.1-8b-instant"
CONTEXT_WINDOW = 131072  # 128K context support

# Initialize Groq LLM
llm = Groq(
    model=MODEL_NAME,
    api_key=GROQ_API_KEY,
    context_window=CONTEXT_WINDOW
)

# Set as global LLM for LlamaIndex
Settings.llm = llm

print(f"✓ Groq LLM initialized: {MODEL_NAME}")
print(f"  Context window: {CONTEXT_WINDOW} tokens")

✓ Groq LLM initialized: llama-3.1-8b-instant
  Context window: 131072 tokens


### Helper Functions

In [5]:
def estimate_tokens(text):
    """
    Rough estimate of tokens in text (4 chars ≈ 1 token for English).
    
    Args:
        text: Input text string
    
    Returns:
        Estimated token count
    """
    return len(text) // 4


def limit_context(context, max_length=15000, verbose=True):
    """
    Limit context length to avoid Groq API token rate limits.
    
    Args:
        context: The full context string from retrieved documents
        max_length: Maximum character limit (default: 15,000 chars ≈ 3,750 tokens)
        verbose: Whether to print warning when truncating
    
    Returns:
        Truncated context string with notification if truncated
    
    Note:
        Groq free tier has 6,000 TPM (tokens per minute) limit for llama-3.1-8b-instant.
        Keep context under 4,000 tokens to leave room for prompt + response.
    """
    if len(context) > max_length:
        if verbose:
            est_before = estimate_tokens(context)
            est_after = estimate_tokens(context[:max_length])
            print(f"⚠ Context truncated from {len(context):,} to {max_length:,} chars")
            print(f"  (est. tokens: {est_before:,} → {est_after:,} to fit 6K TPM limit)")
        return context[:max_length] + "\n\n[Context truncated to fit token limits]"
    return context


def build_context_from_results(results, max_length=15000):
    """
    Build context from ChromaDB query results with automatic length limiting.
    
    Args:
        results: ChromaDB query results object
        max_length: Maximum character limit for final context
    
    Returns:
        Tuple of (context_string, num_docs)
    """
    context_docs = results["documents"][0]
    context = "\n\n".join(context_docs)
    context = limit_context(context, max_length)
    return context, len(context_docs)

### Basic Prompt Structure

> **Key Components of a Good Prompt:**
> 1. **Role** - Define the AI's persona (e.g., "You are a legal assistant")
> 2. **Context** - Provide relevant background information
> 3. **Task** - Clearly state what needs to be done
> 4. **Constraints** - Set boundaries and rules
> 5. **Format** - Specify output structure

In [6]:
# Simple legal QA prompt template
LEGAL_QA_PROMPT = """
You are a legal assistant.

Context:
{context_str}

Question: {query_str}

Instructions:
1. Answer ONLY from context
2. Cite specific sections
3. Say "I don't know" if not in context

Answer:
"""

# Test with sample context
sample_context = """
The Forest Conservation Act, 1980 prohibits the use of forest land for non-forest 
purposes without prior approval from the Central Government. Section 2 states that 
no state government or authority can de-reserve reserved forests or use forest land 
for any other purpose without central approval.
"""

sample_query = "What approval is needed to use forest land for non-forest purposes?"

# Format the prompt
formatted_prompt = LEGAL_QA_PROMPT.format(
    context_str=sample_context,
    query_str=sample_query
)

# Generate response
start_time = time.time()
response = llm.complete(formatted_prompt)
elapsed = time.time() - start_time

print(f"Query: {sample_query}\n")
print(f"Response:\n{response}")
print(f"\n⏱ Response time: {elapsed:.2f} seconds")

Query: What approval is needed to use forest land for non-forest purposes?

Response:
According to the context, prior approval from the Central Government is needed to use forest land for non-forest purposes. This is stated in Section 2 of the Forest Conservation Act, 1980.

⏱ Response time: 0.88 seconds


### Prompt Templates - Different Styles

In [7]:
# Template 1: Basic RAG Prompt
BASIC_RAG_PROMPT = """
You are a legal document assistant. Use the provided context to answer the question.

Context:
{context}

Question: {question}

Answer:
"""

# Template 2: Constrained RAG Prompt (strict rules)
CONSTRAINED_RAG_PROMPT = """
You are a precise legal assistant. Answer questions using ONLY the provided context.

Context:
{context}

Question: {question}

Instructions:
1. Answer ONLY based on the provided context - do not use external knowledge
2. Cite specific sections, paragraphs, or sources when available
3. If the answer is not in the context, state "I don't have enough information in the provided context"
4. Do not make assumptions or inferences beyond what is explicitly stated
5. Keep responses concise but complete

Answer:
"""

# Template 3: Few-Shot Prompt (with examples)
FEW_SHOT_PROMPT = """
You are a legal assistant. Answer questions based on the provided context.

Context:
{context}

Question: {question}

Examples of good answers:
- If context mentions "Section 5 requires approval", answer: "According to Section 5, approval is required."
- If context doesn't contain the answer, say: "The provided context does not contain information about this."

Instructions:
1. Base your answer strictly on the context
2. Cite specific sections or sources
3. If information is not in context, clearly state that

Answer:
"""

print("✓ Defined 3 prompt templates:")
print("  1. BASIC_RAG_PROMPT - Simple and direct")
print("  2. CONSTRAINED_RAG_PROMPT - Strict rules for accuracy")
print("  3. FEW_SHOT_PROMPT - Includes examples for guidance")

✓ Defined 3 prompt templates:
  1. BASIC_RAG_PROMPT - Simple and direct
  2. CONSTRAINED_RAG_PROMPT - Strict rules for accuracy
  3. FEW_SHOT_PROMPT - Includes examples for guidance


### Compare Prompt Templates

In [8]:
# Test all three templates with the same query
test_context = """
The Supreme Court in State of Telangana v. Mohd. Abdul Qasim (2024) 6 SCC 461 
emphasized that forests serve the Earth by regulating carbon emissions, aiding in 
soil conservation, and regulating the water cycle. The Court held that preserving 
the ecosystem is a constitutional duty under Article 48A and Article 51A(g).

Section 2 of the Forest Conservation Act, 1980 prohibits de-reservation of reserved 
forests without Central Government approval. The penalty for violation is 
imprisonment up to 15 days and fine as prescribed.
"""

test_query = "What is the penalty for violating the Forest Conservation Act?"

templates = [
    ("Basic RAG", BASIC_RAG_PROMPT),
    ("Constrained RAG", CONSTRAINED_RAG_PROMPT),
    ("Few-Shot", FEW_SHOT_PROMPT)
]

print(f"Testing prompts with: {test_query}\n")
print("=" * 60)

for name, template in templates:
    print(f"\n{name} Prompt:")
    print("-" * 40)
    
    formatted = template.format(context=test_context, question=test_query)
    
    start = time.time()
    response = llm.complete(formatted)
    elapsed = time.time() - start
    
    print(f"Response: {str(response)[:200]}...")
    print(f"⏱ Time: {elapsed:.2f}s")

Testing prompts with: What is the penalty for violating the Forest Conservation Act?


Basic RAG Prompt:
----------------------------------------
Response: According to the provided context, the penalty for violating the Forest Conservation Act is imprisonment up to 15 days and a fine as prescribed....
⏱ Time: 0.18s

Constrained RAG Prompt:
----------------------------------------
Response: The penalty for violating the Forest Conservation Act is imprisonment up to 15 days and a fine as prescribed. 

Source: Section 2 of the Forest Conservation Act, 1980...
⏱ Time: 0.59s

Few-Shot Prompt:
----------------------------------------
Response: According to the context, the penalty for violating the Forest Conservation Act is imprisonment up to 15 days and a fine as prescribed. This information is mentioned in the context as follows: "The pe...
⏱ Time: 0.19s


### Chain-of-Thought Prompting

> **Chain-of-Thought (CoT):** Encourage the LLM to reason step-by-step before giving a final answer. This improves accuracy for complex legal reasoning tasks.

In [9]:
# Chain-of-Thought prompt for legal analysis
COT_PROMPT = """
You are a legal analyst. Analyze the question step-by-step using the provided context.

Context:
{context}

Question: {question}

Instructions:
Think through this systematically:
1. Identify the key legal issue or question
2. Find relevant sections in the context that address this issue
3. Analyze what those sections say about the issue
4. Draw a conclusion based ONLY on the context
5. Cite specific sections or cases

Format your response as:
Analysis: [Your step-by-step reasoning]
Conclusion: [Final answer with citations]

Answer:
"""

# Test with a complex legal question
complex_query = "What is the constitutional basis for forest protection in India?"

formatted_cot = COT_PROMPT.format(context=test_context, question=complex_query)

print(f"Query: {complex_query}\n")
print("Chain-of-Thought Response:")
print("-" * 50)

start_time = time.time()
response = llm.complete(formatted_cot)
elapsed = time.time() - start_time

print(response)
print(f"\n⏱ Response time: {elapsed:.2f} seconds")

Query: What is the constitutional basis for forest protection in India?

Chain-of-Thought Response:
--------------------------------------------------
Analysis:

1. **Identify the key legal issue or question**: The key legal issue or question is the constitutional basis for forest protection in India.

2. **Find relevant sections in the context that address this issue**: The relevant sections in the context are:
   - Article 48A of the Indian Constitution, which states that "the State shall endeavour to protect and improve the environment and to safeguard the forests and wild life of the country."
   - Article 51A(g) of the Indian Constitution, which states that "it shall be the duty of every citizen of India to protect and improve the natural environment including forests, lakes, rivers and wild life, and to have compassion for living creatures."
   - Section 2 of the Forest Conservation Act, 1980, which prohibits de-reservation of reserved forests without Central Government approval.

### Load Context from ChromaDB

In [10]:
# Connect to Docker ChromaDB (from Notebook 05)
chroma_client = chromadb.HttpClient(host="localhost", port="8000")

# Get collection
collection = chroma_client.get_or_create_collection(name="legal-docs")

print(f"✓ Connected to ChromaDB collection: {collection.name}")
print(f"  Total documents: {collection.count()}")

✓ Connected to ChromaDB collection: legal-docs
  Total documents: 4


### Retrieve Relevant Context

In [11]:
# Query to search for
query = "What are the forest conservation issues?"

# Generate query embedding
query_embedding = model.encode([query]).tolist()

# Retrieve top 2 documents
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

# Build context with helper function
context, num_docs = build_context_from_results(results, max_length=15000)

print(f"Query: {query}\n")
print(f"Retrieved {num_docs} relevant documents")
print(f"Context length: {len(context):,} characters")

⚠ Context truncated from 42,273 to 15,000 chars
  (est. tokens: 10,568 → 3,750 to fit 6K TPM limit)
Query: What are the forest conservation issues?

Retrieved 2 relevant documents
Context length: 15,041 characters


### Generate Answer with RAG Prompt

In [12]:
# Use the constrained prompt for better accuracy
formatted_prompt = CONSTRAINED_RAG_PROMPT.format(context=context, question=query)

# Estimate tokens before calling API
est_tokens = estimate_tokens(formatted_prompt)
print(f"Estimated prompt size: {est_tokens:,} tokens")
print(f"(Groq free tier limit: 6,000 TPM for llama-3.1-8b-instant)\n")

# Generate response
start_time = time.time()
response = llm.complete(formatted_prompt)
elapsed = time.time() - start_time

print(f"Question: {query}\n")
print(f"Answer:\n{response}")
print(f"\n⏱ Response time: {elapsed:.2f} seconds")

Estimated prompt size: 3,893 tokens
(Groq free tier limit: 6,000 TPM for llama-3.1-8b-instant)

Question: What are the forest conservation issues?

Answer:
The forest conservation issues mentioned in the context are:

1. **Encroachment of forest areas**: The context mentions that almost 13,000 sq. km of forest area is under encroachment (para 5).
2. **Depletion of forest cover**: The context states that the forest cover in India is not adequate, with only 21.76% of the total landmass being forested (para 5).
3. **Destruction of forest areas**: The context mentions that the exploitation of forest areas by tea estates, such as the Bombay Burma Trading Corporation Limited, has been going on for over 95 years (para 10).
4. **Lack of restoration**: The context states that the High Court's earlier directions to restore the forest areas have become inadequate for meaningful restoration and rehabilitation (para 23).
5. **Need for scientific survey**: The context mentions that a scientific surv

### RAG with Streaming Response

In [13]:
# New query for streaming demo
stream_query = "What is the role of the Supreme Court in forest protection?"

# Retrieve context
query_embedding = model.encode([stream_query]).tolist()
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2,
    include=["documents"]
)
context, _ = build_context_from_results(results)

# Format prompt
formatted_prompt = CONSTRAINED_RAG_PROMPT.format(context=context, question=stream_query)

print(f"Query: {stream_query}\n")
print("Streaming response:")
print("-" * 50)

# Stream tokens
start_time = time.time()
for chunk in llm.stream_complete(formatted_prompt):
    print(chunk.delta, end="")

elapsed = time.time() - start_time
print(f"\n-" * 50)
print(f"⏱ Streaming completed in {elapsed:.2f} seconds")

⚠ Context truncated from 42,273 to 15,000 chars
  (est. tokens: 10,568 → 3,750 to fit 6K TPM limit)
Query: What is the role of the Supreme Court in forest protection?

Streaming response:
--------------------------------------------------
The Supreme Court plays a crucial role in forest protection. According to the context, the Court has:

1. Issued mandatory directions to the States and other authorities to ensure that the forest cover is maintained/restored (Paragraph 5).
2. Taken up the issue of removal of encroachments and restoration of forests, which were left inconclusive by the High Court (Paragraph 6).
3. Quoted excerpts from its own judgment in State of Telangana and Others v. Mohd. Abdul Qasim (Dead) Per LRs (2024) 6 SCC 461, emphasizing the importance of forest protection (Paragraph 8).
4. Requested the learned Solicitor General and Amicus Curiae to assist the Court in this matter, and received commitments from the Central Government to comply with the Court's directions fo

### Prompt Engineering Best Practices

In [14]:
# Summary of best practices for legal RAG prompts
BEST_PRACTICES = """
✓ Prompt Engineering Best Practices for Legal RAG
================================================

1. ROLE DEFINITION
   - Clearly define the AI's role: "You are a legal assistant"
   - Set expectations for expertise level and tone

2. CONTEXT MANAGEMENT
   - Provide relevant context from retrieved documents
   - Limit context to fit token limits (15K chars ≈ 3,750 tokens)
   - Use clear delimiters between context and question

3. CONSTRAINTS & INSTRUCTIONS
   - "Answer ONLY from context" - prevents hallucination
   - "Cite specific sections" - improves credibility
   - "Say 'I don't know' if not in context" - honest responses

4. OUTPUT FORMAT
   - Specify structure: "Analysis → Conclusion"
   - Request citations: "Include section numbers"
   - Define length: "Keep responses concise"

5. FEW-SHOT EXAMPLES
   - Show 1-2 examples of good answers
   - Demonstrate citation style
   - Show how to handle missing information

6. CHAIN-OF-THOUGHT
   - For complex questions, request step-by-step reasoning
   - Improves accuracy for legal analysis
   - Makes reasoning transparent

7. TOKEN EFFICIENCY
   - Estimate tokens before API call (4 chars ≈ 1 token)
   - Stay under rate limits (6,000 TPM for free tier)
   - Truncate context if needed

8. ITERATIVE REFINEMENT
   - Test prompts with multiple queries
   - Compare responses across templates
   - Refine based on output quality
"""

print(BEST_PRACTICES)


✓ Prompt Engineering Best Practices for Legal RAG

1. ROLE DEFINITION
   - Clearly define the AI's role: "You are a legal assistant"
   - Set expectations for expertise level and tone

2. CONTEXT MANAGEMENT
   - Provide relevant context from retrieved documents
   - Limit context to fit token limits (15K chars ≈ 3,750 tokens)
   - Use clear delimiters between context and question

3. CONSTRAINTS & INSTRUCTIONS
   - "Answer ONLY from context" - prevents hallucination
   - "Cite specific sections" - improves credibility
   - "Say 'I don't know' if not in context" - honest responses

4. OUTPUT FORMAT
   - Specify structure: "Analysis → Conclusion"
   - Request citations: "Include section numbers"
   - Define length: "Keep responses concise"

5. FEW-SHOT EXAMPLES
   - Show 1-2 examples of good answers
   - Demonstrate citation style
   - Show how to handle missing information

6. CHAIN-OF-THOUGHT
   - For complex questions, request step-by-step reasoning
   - Improves accuracy for legal a

### Test Multiple Queries

In [15]:
# Test the RAG system with multiple queries
test_queries = [
    "What is the Forest Conservation Act?",
    "What are the penalties for forest law violations?",
    "What is the significance of tiger reserves?"
]

print("Testing RAG system with multiple queries\n")
print("=" * 60)

for i, test_query in enumerate(test_queries, 1):
    print(f"\n{i}. Query: {test_query}")
    print("-" * 40)
    
    # Retrieve context
    query_embedding = model.encode([test_query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=1,
        include=["documents"]
    )
    context, _ = build_context_from_results(results, max_length=10000)
    
    # Generate response
    formatted = CONSTRAINED_RAG_PROMPT.format(context=context, question=test_query)
    response = llm.complete(formatted)
    
    print(f"Answer: {str(response)[:250]}...")

Testing RAG system with multiple queries


1. Query: What is the Forest Conservation Act?
----------------------------------------
⚠ Context truncated from 20,684 to 10,000 chars
  (est. tokens: 5,171 → 2,500 to fit 6K TPM limit)
Answer: The Forest Conservation Act is mentioned in the context as follows:

"...as per Section 2 of the Forest Conservation Act, 1980, prior approval of the Government of India is required for assigning use of forest land for non-forestry purposes, by way o...

2. Query: What are the penalties for forest law violations?
----------------------------------------
⚠ Context truncated from 20,684 to 10,000 chars
  (est. tokens: 5,171 → 2,500 to fit 6K TPM limit)
Answer: I don't have enough information in the provided context regarding the penalties for forest law violations. The context mentions the importance of preserving forests, the need to restore degraded forest areas, and the consequences of depleting forest ...

3. Query: What is the significance of tiger 